<a href="https://colab.research.google.com/github/abilashkannanv/AIML/blob/main/AgentStreamlit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Install necessary libraries

First, we need to install `streamlit` to create the web application and `PyPDF2` to handle PDF files.

In [ ]:
!pip install streamlit PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 47.7 MB/s eta 0:00:00


In [ ]:
# Install langchain for text splitters
!pip install langchain

In [ ]:
# Upgrade langchain to ensure the latest version and correct module structure
!pip install --upgrade langchain

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.3/114.3 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.3/234.3 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 1.2 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1
  Attempting uninstall: langgraph-checkpoint
    Found existing installation: langgraph-checkpoint 4.0.2
    Uninstalling langgraph-checkpoint-4.0.2:
      Successfully uninstalled langgraph-checkpoint-4.0.2
  Attempting uninstall: langgraph-prebuilt
    Found existing installation: langgraph-prebuilt 1.0.10
    Uninstalling langgraph-prebuilt-1.0.10:
      Successfully uninstalled langgraph-prebuilt-1.0.10
  Attempt

### Streamlit Application Code

Now, let's write the Streamlit application. This code will:

1.  Define the target directory (`/content`).
2.  List all files in that directory.
3.  For each file:
    *   Read its content.
    *   Extract metadata (file name, page count for PDFs, author).
    *   Calculate the word count.
4.  Display the gathered information in an organized manner.

In [ ]:
import streamlit as st
import os
import PyPDF2

# Define the content directory
CONTENT_DIR = '/content'

st.set_page_config(layout="wide", page_title="File Analyzer App")
st.title("File Content and Metadata Analyzer")

def get_file_info(file_path):
    file_name = os.path.basename(file_path)
    file_extension = os.path.splitext(file_name)[1].lower()
    word_count = 0
    page_count = "N/A"
    author = "N/A"

    try:
        if file_extension == '.pdf':
            with open(file_path, 'rb') as f:
                reader = PyPDF2.PdfReader(f)
                page_count = len(reader.pages)
                if reader.metadata and '/Author' in reader.metadata:
                    author = reader.metadata['/Author']

                text_content = ''
                for page_num in range(page_count):
                    text_content += reader.pages[page_num].extract_text() or ''
                word_count = len(text_content.split())
        elif file_extension in ['.txt', '.md']:
            with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                content = f.read()
                word_count = len(content.split())
        else:
            # For other file types, we can still get basic info
            with open(file_path, 'rb') as f:
                content = f.read()
                # Approximate word count for non-text files, or just leave as 0
                word_count = 0 # Cannot reliably count words for binary files

    except Exception as e:
        st.warning(f"Could not process file {file_name}: {e}")
        return {
            "file_name": file_name,
            "extension": file_extension,
            "word_count": "Error",
            "page_count": "Error",
            "author": "Error"
        }

    return {
        "file_name": file_name,
        "extension": file_extension,
        "word_count": word_count,
        "page_count": page_count,
        "author": author
    }


st.header("Files in `/content` directory:")

files_in_content = [f for f in os.listdir(CONTENT_DIR) if os.path.isfile(os.path.join(CONTENT_DIR, f))]

if not files_in_content:
    st.write(f"No files found in the '{CONTENT_DIR}' directory.")
else:
    st.subheader("Processing file information...")
    file_data = []
    for filename in files_in_content:
        full_path = os.path.join(CONTENT_DIR, filename)
        info = get_file_info(full_path)
        file_data.append(info)

    st.success("Files processed!")

    for data in file_data:
        st.subheader(f"File: {data['file_name']}")
        st.write(f"- **Type:** {data['extension']}")
        st.write(f"- **Total Word Count:** {data['word_count']}")
        if data['extension'] == '.pdf':
            st.write(f"- **Pages:** {data['page_count']}")
            st.write(f"- **Author:** {data['author']}")

st.markdown("--- ")
st.info("To run this Streamlit app, save the code as a Python file (e.g., `app.py`) and then execute `!streamlit run app.py` in a new Colab cell, or download it and run locally.")


2026-05-16 06:43:47.126 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-16 06:43:47.127 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-16 06:43:47.269 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-05-16 06:43:47.270 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-16 06:43:47.272 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-16 06:43:47.273 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-16 06:43:47.275 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn

DeltaGenerator()

### How to run the Streamlit application

To view the Streamlit app, you will need to save the above code into a Python file (e.g., `app.py`) and then run it using the `streamlit` command. Here's how you can do it:

1.  **Save the code to a file:** The previous cell generates the Streamlit app code. You can copy it into a file named `app.py` in your Colab environment.
2.  **Run the Streamlit app:** Execute the following command in a new Colab cell. This will start the Streamlit server, and it will provide a public URL you can click to view your app.

    ```bash
    !streamlit run app.py & npx localtunnel --port 8501
    ```

    Wait a few moments after running the command, and you'll see a public URL. Click on that URL to interact with your Streamlit application.

In [ ]:
with open('app.py', 'w') as f:
    f.write("""import streamlit as st
import os
import PyPDF2

# Define the content directory
CONTENT_DIR = '/content'

st.set_page_config(layout="wide", page_title="File Analyzer App")
st.title("File Content and Metadata Analyzer")

def get_file_info(file_path):
    file_name = os.path.basename(file_path)
    file_extension = os.path.splitext(file_name)[1].lower()
    word_count = 0
    page_count = "N/A"
    author = "N/A"

    try:
        if file_extension == '.pdf':
            with open(file_path, 'rb') as f:
                reader = PyPDF2.PdfReader(f)
                page_count = len(reader.pages)
                if reader.metadata and '/Author' in reader.metadata:
                    author = reader.metadata['/Author']

                text_content = ''
                for page_num in range(page_count):
                    text_content += reader.pages[page_num].extract_text() or ''
                word_count = len(text_content.split())
        elif file_extension in ['.txt', '.md']:
            with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                content = f.read()
                word_count = len(content.split())
        else:
            # For other file types, we can still get basic info
            with open(file_path, 'rb') as f:
                content = f.read()
                # Approximate word count for non-text files, or just leave as 0
                word_count = 0 # Cannot reliably count words for binary files

    except Exception as e:
        st.warning(f"Could not process file {file_name}: {e}")
        return {
            "file_name": file_name,
            "extension": file_extension,
            "word_count": "Error",
            "page_count": "Error",
            "author": "Error"
        }

    return {
        "file_name": file_name,
        "extension": file_extension,
        "word_count": word_count,
        "page_count": page_count,
        "author": author
    }


st.header("Files in `/content` directory:")

files_in_content = [f for f in os.listdir(CONTENT_DIR) if os.path.isfile(os.path.join(CONTENT_DIR, f))]

if not files_in_content:
    st.write(f"No files found in the '{CONTENT_DIR}' directory.")
else:
    st.subheader("Processing file information...")
    file_data = []
    for filename in files_in_content:
        full_path = os.path.join(CONTENT_DIR, filename)
        info = get_file_info(full_path)
        file_data.append(info)

    st.success("Files processed!")

    for data in file_data:
        st.subheader(f"File: {data['file_name']}")
        st.write(f"- **Type:** {data['extension']}")
        st.write(f"- **Total Word Count:** {data['word_count']}")
        if data['extension'] == '.pdf':
            st.write(f"- **Pages:** {data['page_count']}")
            st.write(f"- **Author:** {data['author']}")

st.markdown("--- ")
st.info("To run this Streamlit app, save the code as a Python file (e.g., `app.py`) and then execute `!streamlit run app.py` in a new Colab cell, or download it and run locally.")
""")
print("Streamlit app saved to app.py")

Streamlit app saved to app.py


In [35]:
!pip install langchain-text-splitters
from langchain_text_splitters import CharacterTextSplitter

text = "This is a long text that needs splitting."

splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=100,
    chunk_overlap=20
)

chunks = splitter.split_text(text)

print(chunks)

['This is a long text that needs splitting.']


In [65]:
with open('app.py', 'w') as f:
    f.write("""import streamlit as st
import os
import PyPDF2
import shutil

from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from google.colab import userdata
from langchain_community.vectorstores import FAISS
from langchain.chains.retrieval_qa import RetrievalQA # Corrected import path
from langchain.prompts import PromptTemplate

# Set OpenAI API key from Colab secrets
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# Define the content directory and vector store path
CONTENT_DIR = '/content'
VECTOR_STORE_PATH = "faiss_index"

st.set_page_config(layout="wide", page_title="File Analyzer App")
st.title("File Content and Metadata Analyzer and RAG Assistant")

# --- Helper Functions ---
def get_file_info(file_path):
    file_name = os.path.basename(file_path)
    file_extension = os.path.splitext(file_name)[1].lower()
    word_count = 0
    page_count = "N/A"
    author = "N/A"

    try:
        if file_extension == '.pdf':
            with open(file_path, 'rb') as f:
                reader = PyPDF2.PdfReader(f)
                page_count = len(reader.pages)
                if reader.metadata and '/Author' in reader.metadata:
                    author = reader.metadata.get('/Author', 'N/A')

                text_content = ''
                for page_num in range(page_count):
                    text_content += reader.pages[page_num].extract_text() or ''
                word_count = len(text_content.split())
        elif file_extension in ['.txt', '.md']:
            with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                content = f.read()
                word_count = len(content.split())
        else:
            word_count = 0

    except Exception as e:
        st.warning(f"Could not process file {file_name}: {e}")
        return {
            "file_name": file_name,
            "extension": file_extension,
            "word_count": "Error",
            "page_count": "Error",
            "author": "Error"
        }

    return {
        "file_name": file_name,
        "extension": file_extension,
        "word_count": word_count,
        "page_count": page_count,
        "author": author
    }

# --- Display File Information ---
st.header("Files in `/content` directory:")
files_in_content = [f for f in os.listdir(CONTENT_DIR) if os.path.isfile(os.path.join(CONTENT_DIR, f))]

if not files_in_content:
    st.write(f"No files found in the '{CONTENT_DIR}' directory.")
else:
    st.subheader("Processing file information...")
    file_data = []
    for filename in files_in_content:
        full_path = os.path.join(CONTENT_DIR, filename)
        info = get_file_info(full_path)
        file_data.append(info)

    st.success("Files processed!")

    for data in file_data:
        st.subheader(f"File: {data['file_name']}")
        st.write(f"- **Type:** {data['extension']}")
        st.write(f"- **Total Word Count:** {data['word_count']}")
        if data['extension'] == '.pdf':
            st.write(f"- **Pages:** {data['page_count']}")
            st.write(f"- **Author:** {data['author']}")

st.markdown("--- ")

# --- RAG Setup: Document Embeddings ---
st.header("RAG Setup: Document Embeddings")

# Initialize LLM and Embeddings once
llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0.0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = None
all_loaded_documents = [] # Store all loaded documents for the filter tool

if os.path.exists(VECTOR_STORE_PATH) and os.path.isdir(VECTOR_STORE_PATH):
    st.success(f"FAISS Vector Store found at '{VECTOR_STORE_PATH}'. Ready for RAG.")
    try:
        vectorstore = FAISS.load_local(VECTOR_STORE_PATH, embeddings, allow_dangerous_deserialization=True)
        # If vectorstore is loaded, reload all_loaded_documents for filter tool
        st.info("Reloading documents for filter tool functionality...")
        for file_name in files_in_content:
            full_path = os.path.join(CONTENT_DIR, file_name)
            try:
                if file_name.lower().endswith('.pdf'):
                    loader = PyPDFLoader(full_path, silent_errors=True)
                    all_loaded_documents.extend(loader.load())
                elif file_name.lower().endswith(('.txt', '.md')):
                    loader = TextLoader(full_path, encoding='utf-8', autodetect_encoding=True)
                    all_loaded_documents.extend(loader.load())
            except Exception as e:
                st.warning(f"Could not reload document {file_name} for filter tool: {e}")

    except Exception as e:
        st.error(f"Error loading FAISS vector store: {e}. Please regenerate it.")
        if st.button("Delete corrupted vector store and regenerate"): # Key added
            shutil.rmtree(VECTOR_STORE_PATH)
            st.experimental_rerun()
    if st.button("Delete existing vector store and re-generate", key="delete_vectorstore_button"): # Key added
        shutil.rmtree(VECTOR_STORE_PATH)
        st.experimental_rerun()
else:
    st.info("No FAISS Vector Store found. Generate embeddings from documents in `/content`.")
    if st.button("Generate Embeddings for all documents"): # Key added
        with st.spinner("Loading and splitting documents..."):
            for file_name in files_in_content:
                full_path = os.path.join(CONTENT_DIR, file_name)
                try:
                    if file_name.lower().endswith('.pdf'):
                        loader = PyPDFLoader(full_path, silent_errors=True)
                        all_loaded_documents.extend(loader.load())
                    elif file_name.lower().endswith(('.txt', '.md')):
                        loader = TextLoader(full_path, encoding='utf-8', autodetect_encoding=True)
                        all_loaded_documents.extend(loader.load())
                    else:
                        st.warning(f"Skipping unsupported file type for embedding: {file_name}")
                except Exception as e:
                    st.warning(f"Could not load document {file_name} for embedding: {e}")

            if not all_loaded_documents:
                st.warning("No documents could be loaded for embedding. Ensure `/content` contains supported file types (PDF, TXT, MD).")
            else:
                st.success(f"Loaded {len(all_loaded_documents)} documents for embedding.")

                text_splitter = CharacterTextSplitter(
                    chunk_size=1000,
                    chunk_overlap=200,
                    length_function=len
                )
                chunks = text_splitter.split_documents(all_loaded_documents)
                st.success(f"Split documents into {len(chunks)} chunks.")

                with st.spinner("Generating embeddings and building FAISS vector store..."):
                    vectorstore = FAISS.from_documents(chunks, embeddings)
                    vectorstore.save_local(VECTOR_STORE_PATH)
                    st.success(f"FAISS Vector Store created and saved to '{VECTOR_STORE_PATH}'.")
                    st.experimental_rerun()

st.markdown("--- ")

# --- RAG Agent Section ---
st.header("RAG Assistant")
user_query = st.text_input("Ask a question about your documents:")

if user_query:

    # --- Tool Definitions ---

    def rag_retriever_tool(query: str):
        st.write("--- ")
        st.info("Calling RAG Retriever Tool...")
        if vectorstore is None:
            st.error("Vector store not initialized. Please generate embeddings first.")
            return
        qa_chain = RetrievalQA.from_chain_type(
            llm=llm,
            chain_type="stuff",
            retriever=vectorstore.as_retriever(),
            return_source_documents=True
        )
        try:
            response = qa_chain.invoke({"query": query})
            st.subheader("Answer:")
            st.write(response["result"])
            st.subheader("Source Documents:")
            for doc in response["source_documents"]:
                st.write(f"- Source: {os.path.basename(doc.metadata.get('source', 'N/A'))}, Page: {doc.metadata.get('page', 'N/A')}")
        except Exception as e:
            st.error(f"Error in RAG Retriever Tool: {e}")

    def filter_tool(query: str):
        st.write("--- ")
        st.info("Calling Filter Tool...")
        if not all_loaded_documents:
            st.warning("Documents not loaded for filtering. Please generate embeddings to populate the document list.")
            rag_retriever_tool(query) # Fallback to standard RAG
            return

        filename_match = None
        # Extract filename: "file:filename.pdf" or "from filename.txt"
        if "file:" in query.lower():
            start_idx = query.lower().find("file:") + len("file:")
            # Take the word immediately after "file:" or up to next space/colon
            filename_parts = query[start_idx:].strip().split(' ')
            filename_match = filename_parts[0].split(':')[0].split(',')[0].strip()
        elif "from " in query.lower():
            start_idx = query.lower().find("from ") + len("from ")
            # Take the word immediately after "from " or up to next space
            filename_parts = query[start_idx:].strip().split(' ')
            filename_match = filename_parts[0].split(',')[0].strip()

        if not filename_match:
            st.warning("Could not identify a filename in the query for the Filter Tool. Please use 'file:filename.ext' or 'from filename.ext'.")
            rag_retriever_tool(query) # Fallback to standard RAG
            return

        st.write(f"Filtering documents for file: **{filename_match}**")
        filtered_docs = [
            doc for doc in all_loaded_documents
            if filename_match.lower() in os.path.basename(doc.metadata.get('source', '')).lower()
        ]

        if not filtered_docs:
            st.warning(f"No documents found matching '{filename_match}'. Falling back to standard RAG.")
            rag_retriever_tool(query)
            return

        st.success(f"Found {len(filtered_docs)} document(s) matching '{filename_match}'.")

        # Create a temporary vector store from filtered documents
        with st.spinner(f"Generating temporary embeddings for '{filename_match}'..."):
            text_splitter = CharacterTextSplitter(
                chunk_size=1000,
                chunk_overlap=200,
                length_function=len
            )
            filtered_chunks = text_splitter.split_documents(filtered_docs)
            if not filtered_chunks:
                st.warning(f"No chunks could be created from filtered document '{filename_match}'. Falling back to standard RAG.")
                rag_retriever_tool(query)
                return

            temp_vectorstore = FAISS.from_documents(filtered_chunks, embeddings)
            st.success(f"Created temporary FAISS vector store for filtered document.")

        # Re-run QA chain with temporary vector store
        filtered_qa_chain = RetrievalQA.from_chain_type(
            llm=llm,
            chain_type="stuff",
            retriever=temp_vectorstore.as_retriever(),
            return_source_documents=True
        )

        try:
            # Remove filename from query for filtered RAG
            processed_query = query.replace(f"file:{filename_match}", "").replace(f"from {filename_match}", "").strip()
            if not processed_query:
                processed_query = "Summarize the document." # Default query if only filename was provided

            response = filtered_qa_chain.invoke({"query": processed_query})
            st.subheader("Answer (Filtered):")
            st.write(response["result"])
            st.subheader("Source Documents (Filtered):")
            for doc in response["source_documents"]:
                st.write(f"- Source: {os.path.basename(doc.metadata.get('source', 'N/A'))}, Page: {doc.metadata.get('page', 'N/A')}")
        except Exception as e:
            st.error(f"Error in Filter Tool RAG: {e}")


    def elaborator_tool(query: str):
        st.write("--- ")
        st.info("Calling Elaborator Tool...")
        elaborate_prompt_template = PromptTemplate(
            template='''You are an assistant that helps to elaborate and rephrase user queries to make them more detailed for a RAG system. The user wants a detailed answer. Rephrase the following query to be more thorough and comprehensive, ensuring it elicits a detailed response, without adding any conversational text:\n\nOriginal Query: {query}\n\nRephrased Query:''',
            input_variables=["query"]
        )
        # Using .invoke() instead of `| llm` which assumes LCEL
        elaborator_chain = elaborate_prompt_template | llm
        try:
            # The `invoke` method for a chain (prompt | llm) returns a message or a string directly if output parser is used.
            # For simple cases like this, .content should work.
            elaborated_query = elaborator_chain.invoke({"query": query}).content
            st.write(f"Original Query: *{query}*")
            st.write(f"Elaborated Query: **{elaborated_query}**")
            st.success("Query elaborated. Now calling RAG Retriever Tool with the elaborated query.")
            rag_retriever_tool(elaborated_query)
        except Exception as e:
            st.error(f"Error in Elaborator Tool: {e}")
            st.warning("Falling back to standard RAG with original query.")
            rag_retriever_tool(query)


    # --- Agent Logic ---
    if st.button("Get Answer", key="get_answer_button_rag"): # Unique key
        if vectorstore is None:
            st.error("Please generate the FAISS vector store first.")
        else:
            lower_query = user_query.lower()
            if "file:" in lower_query or "from " in lower_query:
                filter_tool(user_query)
            elif any(keyword in lower_query for keyword in ["carefully", "detailed", "thoroughly"]):
                elaborator_tool(user_query)
            else:
                rag_retriever_tool(user_query)

st.markdown("--- ")
st.info("To run this Streamlit app, save the code as a Python file (e.g., `app.py`) and then execute `!streamlit run app.py` in a new Colab cell, or download it and run locally.")
""")
print("Streamlit app updated with RAG functionality and saved to app.py")

Streamlit app updated with RAG functionality and saved to app.py


In [66]:
import subprocess
import time
import webbrowser

STREAMLIT_PORT = 8501

# Start Streamlit
streamlit_process = subprocess.Popen(
    ["streamlit", "run", "app.py"]
)

print("Starting Streamlit app...")
time.sleep(5)

# Start LocalTunnel
# Use 'npx localtunnel' instead of just 'lt'
lt_process = subprocess.Popen(
    ["npx", "localtunnel", "--port", str(STREAMLIT_PORT)],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

print("Starting LocalTunnel...")
time.sleep(5)

# Read LocalTunnel output
while True:
    line = lt_process.stdout.readline()

    if "your url is:" in line.lower():
        public_url = line.split("is:")[-1].strip()

        print(f"\nPublic URL: {public_url}")

        # Open in browser
        webbrowser.open(public_url)

        break

# Keep processes alive
try:
    streamlit_process.wait()
except KeyboardInterrupt:
    streamlit_process.terminate()
    lt_process.terminate()

Starting Streamlit app...
Starting LocalTunnel...

Public URL: https://tasty-maps-dream.loca.lt


In [ ]:
# Install necessary libraries for embeddings and vector store
!pip install faiss-cpu tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 54.9 MB/s eta 0:00:00


In [20]:
with open('app.py', 'r') as f:
    app_content = f.read()
print(app_content)

import streamlit as st
import os
import PyPDF2
import shutil
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from google.colab import userdata
from langchain_community.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

# Set OpenAI API key from Colab secrets
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# Define the content directory and vector store path
CONTENT_DIR = '/content'
VECTOR_STORE_PATH = "faiss_index"

st.set_page_config(layout="wide", page_title="File Analyzer App")
st.title("File Content and Metadata Analyzer and RAG Assistant")

# --- Helper Functions ---
def get_file_info(file_path):
    file_name = os.path.basename(file_path)
    file_extension = os.path.splitext(file_name)[1].lower()
    word_count = 0
    page_count = "N/A"
    author = "N/A"

    try

### Set OpenAI API Key and Text Processing Utilities

This section sets up your OpenAI API key and provides helper functions to extract text from files and split it into chunks, which are necessary steps before generating embeddings.

### Set OpenAI API Key

To use `langchain_openai.OpenAIEmbeddings`, ensure your OpenAI API key is securely stored in Colab secrets with the name `OPENAI_API_KEY`. You can set it using `os.environ`:

```python
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
```

In [18]:
import os
import openai
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
openai.api_key = "voc-262423617156170395256969020d2c765580.35946547"

### Restarting Runtime and Reinstalling Libraries

As requested, I will now reinstall all necessary libraries to ensure a clean environment and resolve any potential module path conflicts. Following that, I will update the `app.py` file with the correct import statements for `CharacterTextSplitter` and `RetrievalQA`.

In [67]:
# Install all necessary libraries
!pip install --upgrade streamlit PyPDF2 langchain langchain-text-splitters faiss-cpu tiktoken langchain-openai langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 16.6 MB/s eta 0:00:00
ERROR: Operation cancelled by user
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/operations/check.p

### Update Streamlit Application Code with Corrected Imports

Now, I will rewrite the `app.py` file. I have corrected the import paths for `CharacterTextSplitter` to `from langchain_text_splitters import CharacterTextSplitter` and `RetrievalQA` to `from langchain.chains.retrieval_qa import RetrievalQA` to address the `ModuleNotFoundError` previously encountered.

In [68]:
with open('app.py', 'w') as f:
    f.write("""import streamlit as st
import os
import PyPDF2
import shutil

from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import CharacterTextSplitter # Corrected import path for CharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from google.colab import userdata
from langchain_community.vectorstores import FAISS
from langchain.chains.retrieval_qa import RetrievalQA # Corrected import path for RetrievalQA
from langchain.prompts import PromptTemplate

# Set OpenAI API key from Colab secrets
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# Define the content directory and vector store path
CONTENT_DIR = '/content'
VECTOR_STORE_PATH = "faiss_index"

st.set_page_config(layout="wide", page_title="File Analyzer App")
st.title("File Content and Metadata Analyzer and RAG Assistant")

# --- Helper Functions ---
def get_file_info(file_path):
    file_name = os.path.basename(file_path)
    file_extension = os.path.splitext(file_name)[1].lower()
    word_count = 0
    page_count = "N/A"
    author = "N/A"

    try:
        if file_extension == '.pdf':
            with open(file_path, 'rb') as f:
                reader = PyPDF2.PdfReader(f)
                page_count = len(reader.pages)
                if reader.metadata and '/Author' in reader.metadata:
                    author = reader.metadata.get('/Author', 'N/A')

                text_content = ''
                for page_num in range(page_count):
                    text_content += reader.pages[page_num].extract_text() or ''
                word_count = len(text_content.split())
        elif file_extension in ['.txt', '.md']:
            with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                content = f.read()
                word_count = len(content.split())
        else:
            word_count = 0

    except Exception as e:
        st.warning(f"Could not process file {file_name}: {e}")
        return {
            "file_name": file_name,
            "extension": file_extension,
            "word_count": "Error",
            "page_count": "Error",
            "author": "Error"
        }

    return {
        "file_name": file_name,
        "extension": file_extension,
        "word_count": word_count,
        "page_count": page_count,
        "author": author
    }

# --- Display File Information ---
st.header("Files in `/content` directory:")
files_in_content = [f for f in os.listdir(CONTENT_DIR) if os.path.isfile(os.path.join(CONTENT_DIR, f))]

if not files_in_content:
    st.write(f"No files found in the '{CONTENT_DIR}' directory.")
else:
    st.subheader("Processing file information...")
    file_data = []
    for filename in files_in_content:
        full_path = os.path.join(CONTENT_DIR, filename)
        info = get_file_info(full_path)
        file_data.append(info)

    st.success("Files processed!")

    for data in file_data:
        st.subheader(f"File: {data['file_name']}")
        st.write(f"- **Type:** {data['extension']}")
        st.write(f"- **Total Word Count:** {data['word_count']}")
        if data['extension'] == '.pdf':
            st.write(f"- **Pages:** {data['page_count']}")
            st.write(f"- **Author:** {data['author']}")

st.markdown("--- ")

# --- RAG Setup: Document Embeddings ---
st.header("RAG Setup: Document Embeddings")

# Initialize LLM and Embeddings once
llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0.0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = None
all_loaded_documents = [] # Store all loaded documents for the filter tool

if os.path.exists(VECTOR_STORE_PATH) and os.path.isdir(VECTOR_STORE_PATH):
    st.success(f"FAISS Vector Store found at '{VECTOR_STORE_PATH}'. Ready for RAG.")
    try:
        vectorstore = FAISS.load_local(VECTOR_STORE_PATH, embeddings, allow_dangerous_deserialization=True)
        # If vectorstore is loaded, reload all_loaded_documents for filter tool
        st.info("Reloading documents for filter tool functionality...")
        for file_name in files_in_content:
            full_path = os.path.join(CONTENT_DIR, file_name)
            try:
                if file_name.lower().endswith('.pdf'):
                    loader = PyPDFLoader(full_path, silent_errors=True)
                    all_loaded_documents.extend(loader.load())
                elif file_name.lower().endswith(('.txt', '.md')):
                    loader = TextLoader(full_path, encoding='utf-8', autodetect_encoding=True)
                    all_loaded_documents.extend(loader.load())
            except Exception as e:
                st.warning(f"Could not reload document {file_name} for filter tool: {e}")

    except Exception as e:
        st.error(f"Error loading FAISS vector store: {e}. Please regenerate it.")
        if st.button("Delete corrupted vector store and regenerate"): # Key added
            shutil.rmtree(VECTOR_STORE_PATH)
            st.experimental_rerun()
    if st.button("Delete existing vector store and re-generate", key="delete_vectorstore_button"): # Key added
        shutil.rmtree(VECTOR_STORE_PATH)
        st.experimental_rerun()
else:
    st.info("No FAISS Vector Store found. Generate embeddings from documents in `/content`.")
    if st.button("Generate Embeddings for all documents"): # Key added
        with st.spinner("Loading and splitting documents..."):
            for file_name in files_in_content:
                full_path = os.path.join(CONTENT_DIR, file_name)
                try:
                    if file_name.lower().endswith('.pdf'):
                        loader = PyPDFLoader(full_path, silent_errors=True)
                        all_loaded_documents.extend(loader.load())
                    elif file_name.lower().endswith(('.txt', '.md')):
                        loader = TextLoader(full_path, encoding='utf-8', autodetect_encoding=True)
                        all_loaded_documents.extend(loader.load())
                    else:
                        st.warning(f"Skipping unsupported file type for embedding: {file_name}")
                except Exception as e:
                    st.warning(f"Could not load document {file_name} for embedding: {e}")

            if not all_loaded_documents:
                st.warning("No documents could be loaded for embedding. Ensure `/content` contains supported file types (PDF, TXT, MD).")
            else:
                st.success(f"Loaded {len(all_loaded_documents)} documents for embedding.")

                text_splitter = CharacterTextSplitter(
                    chunk_size=1000,
                    chunk_overlap=200,
                    length_function=len
                )
                chunks = text_splitter.split_documents(all_loaded_documents)
                st.success(f"Split documents into {len(chunks)} chunks.")

                with st.spinner("Generating embeddings and building FAISS vector store..."):
                    vectorstore = FAISS.from_documents(chunks, embeddings)
                    vectorstore.save_local(VECTOR_STORE_PATH)
                    st.success(f"FAISS Vector Store created and saved to '{VECTOR_STORE_PATH}'.")
                    st.experimental_rerun()

st.markdown("--- ")

# --- RAG Agent Section ---
st.header("RAG Assistant")
user_query = st.text_input("Ask a question about your documents:")

if user_query:

    # --- Tool Definitions ---

    def rag_retriever_tool(query: str):
        st.write("--- ")
        st.info("Calling RAG Retriever Tool...")
        if vectorstore is None:
            st.error("Vector store not initialized. Please generate embeddings first.")
            return
        qa_chain = RetrievalQA.from_chain_type(
            llm=llm,
            chain_type="stuff",
            retriever=vectorstore.as_retriever(),
            return_source_documents=True
        )
        try:
            response = qa_chain.invoke({"query": query})
            st.subheader("Answer:")
            st.write(response["result"])
            st.subheader("Source Documents:")
            for doc in response["source_documents"]:
                st.write(f"- Source: {os.path.basename(doc.metadata.get('source', 'N/A'))}, Page: {doc.metadata.get('page', 'N/A')}")
        except Exception as e:
            st.error(f"Error in RAG Retriever Tool: {e}")

    def filter_tool(query: str):
        st.write("--- ")
        st.info("Calling Filter Tool...")
        if not all_loaded_documents:
            st.warning("Documents not loaded for filtering. Please generate embeddings to populate the document list.")
            rag_retriever_tool(query) # Fallback to standard RAG
            return

        filename_match = None
        # Extract filename: "file:filename.pdf" or "from filename.txt"
        if "file:" in query.lower():
            start_idx = query.lower().find("file:") + len("file:")
            # Take the word immediately after "file:" or up to next space/colon
            filename_parts = query[start_idx:].strip().split(' ')
            filename_match = filename_parts[0].split(':')[0].split(',')[0].strip()
        elif "from " in query.lower():
            start_idx = query.lower().find("from ") + len("from ")
            # Take the word immediately after "from " or up to next space
            filename_parts = query[start_idx:].strip().split(' ')
            filename_match = filename_parts[0].split(',')[0].strip()

        if not filename_match:
            st.warning("Could not identify a filename in the query for the Filter Tool. Please use 'file:filename.ext' or 'from filename.ext'.")
            rag_retriever_tool(query) # Fallback to standard RAG
            return

        st.write(f"Filtering documents for file: **{filename_match}**")
        filtered_docs = [
            doc for doc in all_loaded_documents
            if filename_match.lower() in os.path.basename(doc.metadata.get('source', '')).lower()
        ]

        if not filtered_docs:
            st.warning(f"No documents found matching '{filename_match}'. Falling back to standard RAG.")
            rag_retriever_tool(query)
            return

        st.success(f"Found {len(filtered_docs)} document(s) matching '{filename_match}'.")

        # Create a temporary vector store from filtered documents
        with st.spinner(f"Generating temporary embeddings for '{filename_match}'..."):
            text_splitter = CharacterTextSplitter(
                chunk_size=1000,
                chunk_overlap=200,
                length_function=len
            )
            filtered_chunks = text_splitter.split_documents(filtered_docs)
            if not filtered_chunks:
                st.warning(f"No chunks could be created from filtered document '{filename_match}'. Falling back to standard RAG.")
                rag_retriever_tool(query)
                return

            temp_vectorstore = FAISS.from_documents(filtered_chunks, embeddings)
            st.success(f"Created temporary FAISS vector store for filtered document.")

        # Re-run QA chain with temporary vector store
        filtered_qa_chain = RetrievalQA.from_chain_type(
            llm=llm,
            chain_type="stuff",
            retriever=temp_vectorstore.as_retriever(),
            return_source_documents=True
        )

        try:
            # Remove filename from query for filtered RAG
            processed_query = query.replace(f"file:{filename_match}", "").replace(f"from {filename_match}", "").strip()
            if not processed_query:
                processed_query = "Summarize the document." # Default query if only filename was provided

            response = filtered_qa_chain.invoke({"query": processed_query})
            st.subheader("Answer (Filtered):")
            st.write(response["result"])
            st.subheader("Source Documents (Filtered):")
            for doc in response["source_documents"]:
                st.write(f"- Source: {os.path.basename(doc.metadata.get('source', 'N/A'))}, Page: {doc.metadata.get('page', 'N/A')}")
        except Exception as e:
            st.error(f"Error in Filter Tool RAG: {e}")


    def elaborator_tool(query: str):
        st.write("--- ")
        st.info("Calling Elaborator Tool...")
        elaborate_prompt_template = PromptTemplate(
            template='''You are an assistant that helps to elaborate and rephrase user queries to make them more detailed for a RAG system. The user wants a detailed answer. Rephrase the following query to be more thorough and comprehensive, ensuring it elicits a detailed response, without adding any conversational text:\n\nOriginal Query: {query}\n\nRephrased Query:''',
            input_variables=["query"]
        )
        # Using .invoke() instead of `| llm` which assumes LCEL
        elaborator_chain = elaborate_prompt_template | llm
        try:
            # The `invoke` method for a chain (prompt | llm) returns a message or a string directly if output parser is used.
            # For simple cases like this, .content should work.
            elaborated_query = elaborator_chain.invoke({"query": query}).content
            st.write(f"Original Query: *{query}*")
            st.write(f"Elaborated Query: **{elaborated_query}**")
            st.success("Query elaborated. Now calling RAG Retriever Tool with the elaborated query.")
            rag_retriever_tool(elaborated_query)
        except Exception as e:
            st.error(f"Error in Elaborator Tool: {e}")
            st.warning("Falling back to standard RAG with original query.")
            rag_retriever_tool(query)


    # --- Agent Logic ---
    if st.button("Get Answer", key="get_answer_button_rag"): # Unique key
        if vectorstore is None:
            st.error("Please generate the FAISS vector store first.")
        else:
            lower_query = user_query.lower()
            if "file:" in lower_query or "from " in lower_query:
                filter_tool(user_query)
            elif any(keyword in lower_query for keyword in ["carefully", "detailed", "thoroughly"]):
                elaborator_tool(user_query)
            else:
                rag_retriever_tool(user_query)

st.markdown("--- ")
st.info("To run this Streamlit app, save the code as a Python file (e.g., `app.py`) and then execute `!streamlit run app.py` in a new Colab cell, or download it and run locally.")
""")
print("Streamlit app updated with RAG functionality and saved to app.py")

Streamlit app updated with RAG functionality and saved to app.py
